# Data Validation Quick Check

**Run this notebook FIRST** before training models to ensure your Excel data is clean.

This will:
1. Check how percentages are stored
2. Clean any percentage symbols
3. Validate data quality
4. Save a clean version for model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries loaded!")

In [ ]:
# Load your data
file_path = '../data/raw/Master_One_Sheet.xlsx'

print("Loading data...")
df = pd.read_excel(file_path)

print(f"✅ Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFirst few rows:")
df.head()

## Step 1: Check Percentage Columns

In [ ]:
# Find all percentage columns
pct_cols = [col for col in df.columns if '%' in col]

print(f"Found {len(pct_cols)} percentage columns:")
for col in pct_cols:
    print(f"  • {col}")

# Check first few percentage values
if len(pct_cols) > 0:
    print(f"\nSample values from first percentage column: {pct_cols[0]}")
    print(df[pct_cols[0]].head(10).tolist())
    print(f"\nData type: {df[pct_cols[0]].dtype}")

In [ ]:
# Analyze percentage ranges
print("Percentage Column Ranges:")
print("="*70)

for col in pct_cols[:5]:  # First 5 to avoid too much output
    data = df[col].dropna()
    if len(data) > 0:
        print(f"\n{col}:")
        print(f"  Data type: {df[col].dtype}")
        
        try:
            # Try to get numeric stats
            print(f"  Min:    {data.min()}")
            print(f"  Max:    {data.max()}")
            print(f"  Mean:   {data.mean():.2f}")
            print(f"  Median: {data.median():.2f}")
            
            if data.max() > 10:
                print(f"  📊 Scale: Standard percentage (0-100+)")
            elif data.max() <= 1.5:
                print(f"  📊 Scale: Decimal format (0-1)")
                
        except Exception as e:
            print(f"  ⚠️  Error reading as numeric: {e}")
            print(f"  Sample values: {data.head(3).tolist()}")

## Step 2: Clean Percentage Columns

This removes any '%' symbols and ensures all percentage columns are numeric.

In [ ]:
df_clean = df.copy()

print("Cleaning percentage columns...")
print("="*70)

cleaned_count = 0

for col in pct_cols:
    original_dtype = df_clean[col].dtype
    
    # If column is object/string type, clean it
    if df_clean[col].dtype == 'object':
        print(f"\nCleaning {col}...")
        
        # Remove % symbols and convert to numeric
        df_clean[col] = df_clean[col].astype(str).str.replace('%', '').str.strip()
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
        print(f"  ✓ Converted from {original_dtype} to numeric")
        cleaned_count += 1
    else:
        # Already numeric, just verify
        print(f"✓ {col} - already numeric")

print(f"\n{'='*70}")
print(f"✅ Cleaned {cleaned_count} columns")
print(f"✅ All {len(pct_cols)} percentage columns are now numeric")

## Step 3: Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values Check:")
print("="*70)

missing = df_clean.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print(f"\nFound missing values in {len(missing)} columns:")
    print(missing.head(15))
    print("\n⚠️  Missing values will be filled with 0 during training.")
else:
    print("\n✅ No missing values!")

In [ ]:
# Validate months
print("Month Validation:")
print("="*70)

print(f"\nUnique months: {df_clean['Month'].nunique()}")
print(f"Months: {sorted(df_clean['Month'].unique())}")

if df_clean['Month'].nunique() >= 6:
    print(f"\n✅ Sufficient data ({df_clean['Month'].nunique()} months)")
else:
    print(f"\n⚠️  Only {df_clean['Month'].nunique()} months (recommend 6+)")

In [ ]:
# Check competitor loans (our main target)
print("Competitor Loans Validation:")
print("="*70)

comp_loans = df_clean['Competitor Loans this Month']

print(f"\nStatistics:")
print(comp_loans.describe())

# Visualize distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
comp_loans.hist(bins=30, edgecolor='black')
plt.xlabel('Competitor Loans')
plt.ylabel('Frequency')
plt.title('Distribution of Competitor Loans')

plt.subplot(1, 2, 2)
comp_loans.value_counts().head(10).plot(kind='bar', edgecolor='black')
plt.xlabel('Number of Loans')
plt.ylabel('Count')
plt.title('Top 10 Loan Counts')

plt.tight_layout()
plt.show()

# Show class distribution based on thresholds
print("\nBroker Classification (current month):")
print(f"  Others (< 2 loans):    {(comp_loans < 2).sum():,} brokers")
print(f"  Minnow (2-3 loans):    {((comp_loans >= 2) & (comp_loans < 4)).sum():,} brokers")
print(f"  Dolphin (4-5 loans):   {((comp_loans >= 4) & (comp_loans < 6)).sum():,} brokers")
print(f"  Whale (6+ loans):      {(comp_loans >= 6).sum():,} brokers")

## Step 4: Check for Extreme Percentage Values

Some percentages may be >100% or even >500%. This is OK!

In [ ]:
print("Extreme Value Detection:")
print("="*70)

extreme_summary = []

for col in pct_cols:
    data = df_clean[col].dropna()
    if len(data) > 0:
        max_val = data.max()
        if max_val > 200:  # Flag if >200%
            extreme_summary.append({
                'Column': col,
                'Max Value': f"{max_val:.1f}%",
                'Count >200%': (data > 200).sum()
            })

if extreme_summary:
    extreme_df = pd.DataFrame(extreme_summary)
    print(f"\nFound {len(extreme_summary)} columns with very high percentages (>200%):")
    print(extreme_df.to_string(index=False))
    print("\n✅ This is NORMAL and OK!")
    print("   High percentages (like 900%) can occur in business metrics.")
    print("   Random Forest and Gradient Boosting handle these well.")
else:
    print("\nNo extreme percentages detected (all <200%)")

print("="*70)

## Step 5: Save Cleaned Data

Save the cleaned version for use in model training.

In [ ]:
# Save cleaned data
output_path = '../data/interim/cleaned_broker_data.xlsx'

df_clean.to_excel(output_path, index=False)

print("="*70)
print("✅ DATA CLEANING COMPLETE!")
print("="*70)
print(f"\n💾 Clean data saved to: {output_path}")
print(f"\n📊 Final data shape: {df_clean.shape}")
print("\n✅ You can now use this file for model training!")
print("\nNext steps:")
print("  1. Open 01_broker_classification_pipeline.ipynb")
print("  2. Update the file path to use: '../data/interim/cleaned_broker_data.xlsx'")
print("  3. Run the training notebook")
print("="*70)

## Summary

### What This Notebook Did:

1. ✅ Loaded your Excel data
2. ✅ Identified and cleaned percentage columns
3. ✅ Removed any '%' symbols
4. ✅ Converted all percentages to numeric format
5. ✅ Validated data quality
6. ✅ Saved clean version for modeling

### About High Percentages:

**Why 900% percentages are OK:**
- These represent real business metrics (e.g., growth rates, ratios)
- Machine learning models care about **relationships** between features, not absolute scale
- Tree-based models (Random Forest, Gradient Boosting) are **scale-invariant**
- They work by making decisions like "is this value higher or lower than X?"
- Whether X is 900 or 0.9 doesn't matter - the ranking and relationships stay the same

### Ready for Training!

Your data is now clean and ready. Proceed to the training notebook.